# L0 step 2 · The merge: checkpoint-28 into our 7B (CPU session)

Task **L0**, step 2, from `Essential_task_delegation/Progress_Tracker.md`. A named copy of Cells 1, 2 and 6 of `l0-step2-topup-training.ipynb`, which stays as it ran (session 3b, 26 Oct 2026).

**Why**
- The training finished on 26 Oct: 28 of 28 steps, and `checkpoint-28` is on Hugging Face (`AbrarAlam/disasterm3-qwen25vl7b-qlora`, under `l0s2/checkpoints/`).
- The training notebook's Cell 6 then stopped after 5 seconds: `ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported`.
- Kaggle ships torchao 0.10.0. The peft that Cell 1 upgraded (0.21.2) checks torchao only when it merges into an unquantized (fp16) model. Training loaded the model in 4-bit, so it never reached that check.

**What changes from the training notebook**
- Cell 1 removes torchao, which nothing here uses, and installs peft 0.21.2, the version that trained the adapter, instead of the newest.
- Cell 2 downloads only `checkpoint-28`'s adapter and `trainer_state.json`, not the optimizer state. It checks that the training is complete (28 of 28 steps).
- Cell 3 is the old Cell 6 (fp16, on the CPU, as the first merge was done), plus three checks:
  - exactly the 196 language-model projections the adapter trained change, and nothing else (vision tower, embeddings, norms, biases);
  - no changed weight holds an inf or NaN after the fp16 merge;
  - every file on Hugging Face has the same size as the local copy.
- The weight checks run before anything is pushed, so a failed check leaves Hugging Face unchanged.
- No GPU is needed and nothing is retrained.

**Expectation, fixed before the run** (26 Oct)
- 196 weight tensors change, every one of them a language-model projection.
- The merged model (about 16.6 GB) is on Hugging Face as `AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`, public like our first merged 7B. Set `MERGED_PUBLIC = False` in Cell 2 to keep it private.
- About 30–45 min, mostly the two 16 GB transfers.

**How to run on Kaggle** (any account; a CPU session uses no GPU hours)
1. Settings: Accelerator **None**, internet ON.
2. Secret `HF_TOKEN` (AbrarAlam, **write** access).
3. Save & Run All. There's no smoke test: nothing is pushed until Cell 3's weight checks pass.
4. Output in `/kaggle/working/L0S2-merge/`: `README.md` and `merge_check.json`.
5. Next: `l0-step2-full-size-rescore.ipynb` with `MODEL = "new"`, in parts A, B and C.

In [1]:
# ── Cell 1: Environment: Kaggle's torch and transformers, the training's peft, and no torchao ──
# Session 3b's merge (26 Oct) stopped on "Found an incompatible version of torchao. Found version 0.10.0, but only
# versions above 0.16.0 are supported": peft checks torchao when it merges into an fp16 model. Nothing here uses it.
import importlib.metadata as _md
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft==0.21.2"], check=True)   # the version that trained the adapter

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

assert importlib.util.find_spec("torchao") is None, "torchao is still installed: restart the session and run again"
assert int(_version("transformers").split(".")[0]) >= 5, "written for transformers 5 (Kaggle's 5.16.1 on 26 Oct)"

import psutil
import torch

VERSIONS = {p: _version(p) for p in ("torch", "transformers", "peft", "safetensors", "huggingface_hub")}
RAM_GB = psutil.virtual_memory().total / 1e9
DISK_GB = min(shutil.disk_usage(d).free for d in ("/tmp", str(Path.home()))) / 1e9
print(f"✓ {VERSIONS}, python {sys.version.split()[0]}; torchao: not installed")
print(f"  RAM {RAM_GB:.0f} GB; free disk {DISK_GB:.0f} GB")
assert RAM_GB >= 24, "the fp16 merge holds the 7B in RAM (about 17 GB): use a standard Kaggle session"
assert DISK_GB >= 36, "the starting model and the merged one need about 34 GB of disk"

✓ {'torch': '2.11.0+cu128', 'transformers': '5.16.1', 'peft': '0.21.2', 'safetensors': '0.8.0', 'huggingface_hub': '1.29.0'}, python 3.13.15; torchao: not installed
  RAM 34 GB; free disk 1126 GB


In [2]:
# ── Cell 2: Configuration, Hugging Face access, checkpoint-28's adapter and the starting model ──
import json
import os

from huggingface_hub import HfApi, snapshot_download

MERGED_PUBLIC = True           # like our first merged 7B, whose repo is public
WORK = Path("/kaggle/working/L0S2-merge")
CFG = {
    "task": "L0 step 2: the merge",
    "start_model": "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP",         # our 7B (CSE499A), merged
    "merged_repo": "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP",    # the result; l0-step2-full-size-rescore reads it
    "merged_public": MERGED_PUBLIC,
    "adapter_repo": "AbrarAlam/disasterm3-qwen25vl7b-qlora",            # private
    "hub_dir": "l0s2/checkpoints",                                      # the training's checkpoints in it
    "max_steps": 28,
    "projections": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_targets": 196,       # 28 layers x 7 projections, as train.py asserted
}
WORK.mkdir(parents=True, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Add the HF_TOKEN secret (AbrarAlam, write access)") from e
api = HfApi(token=os.environ["HF_TOKEN"])

# The last checkpoint's adapter and trainer state (a merge doesn't need the optimizer state)
prefix = CFG["hub_dir"] + "/"
steps = sorted({int(f[len(prefix):].split("/")[0].split("-")[1])
                for f in api.list_repo_files(CFG["adapter_repo"]) if f.startswith(prefix + "checkpoint-")})
assert steps and steps[-1] == CFG["max_steps"], f"the last checkpoint is {steps[-1:]}, not checkpoint-{CFG['max_steps']}"
NAME = f"checkpoint-{steps[-1]}"
local = snapshot_download(CFG["adapter_repo"], token=os.environ["HF_TOKEN"],
                          allow_patterns=[f"{prefix}{NAME}/adapter_*", f"{prefix}{NAME}/trainer_state.json"])
ADAPTER = Path(local) / CFG["hub_dir"] / NAME
STATE = json.loads((ADAPTER / "trainer_state.json").read_text(encoding="utf-8"))
ADAPTER_CFG = json.loads((ADAPTER / "adapter_config.json").read_text(encoding="utf-8"))
assert STATE["global_step"] == STATE["max_steps"] == CFG["max_steps"], \
    f"training isn't complete: step {STATE['global_step']} of {STATE['max_steps']}"
assert (ADAPTER / "adapter_model.safetensors").exists(), f"no adapter_model.safetensors in {NAME}"
# One LoRA pair per adapted module. (27 Oct: the config's target_modules can't be counted: given more than 20 module
# names, peft saves the shortest name patterns that pick the same modules, 88 here: q/k/v/o_proj plus each layer's MLP.)
from safetensors import safe_open
with safe_open(str(ADAPTER / "adapter_model.safetensors"), "pt") as f:
    LORA_PAIRS = sum(".lora_A." in k for k in f.keys())
assert LORA_PAIRS == CFG["lora_targets"], f"the adapter holds {LORA_PAIRS} LoRA pairs, not {CFG['lora_targets']}"
LOSSES = [round(e["loss"], 4) for e in STATE["log_history"] if "loss" in e]
print(f"✓ {NAME} restored: step {STATE['global_step']} of {STATE['max_steps']}; LoRA rank {ADAPTER_CFG['r']}, "
      f"alpha {ADAPTER_CFG['lora_alpha']}, {LORA_PAIRS} LoRA pairs ({len(ADAPTER_CFG['target_modules'])} saved name "
      f"patterns); last three losses {LOSSES[-3:]} (session 3b's log: 0.4905, 0.5365, 0.5335)")

START = snapshot_download(CFG["start_model"], token=os.environ["HF_TOKEN"])     # about 16.6 GB
print(f"✓ Starting model cached at {START}")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

✓ checkpoint-28 restored: step 28 of 28; LoRA rank 64, alpha 16, 196 LoRA pairs (88 saved name patterns); last three losses [0.4905, 0.5365, 0.5335] (session 3b's log: 0.4905, 0.5365, 0.5335)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✓ Starting model cached at /root/.cache/huggingface/hub/models--AbrarAlam--disasterm3-qwen2.5vl7b-mergedFP/snapshots/8553f6e620d2b31702bb3d3847e13a3e8c0e3747


In [3]:
# ── Cell 3: Merge on the CPU in fp16, check which weights changed, save, push, and check the upload ──
import gc
import time

from peft import PeftModel
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

def fingerprint(p, chunk=1 << 24):
    """(sum, sum of squares) of a tensor in float64, read in chunks: a merged-in change alters them."""
    flat = p.detach().reshape(-1)
    s = q = 0.0
    for i in range(0, flat.numel(), chunk):
        c = flat[i:i + chunk].to(torch.float64)
        s += float(c.sum())
        q += float((c * c).sum())
    return s, q

t0 = time.time()
print(f"⏳ Merging {NAME} into {CFG['start_model']} (fp16, on the CPU, as the first merge was done)...")
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(START, dtype=torch.float16, device_map="cpu")
# The adapter's targets by train.py's own rule: the 7 projections of every language-model layer, never the vision tower
TARGETS = {f"{n}.weight" for n, _ in base.named_modules()
           if n.rsplit(".", 1)[-1] in CFG["projections"] and "visual" not in n}
assert len(TARGETS) == CFG["lora_targets"], f"expected {CFG['lora_targets']} projections, found {len(TARGETS)}"
before = {n: fingerprint(p) for n, p in base.named_parameters()}
merged = PeftModel.from_pretrained(base, str(ADAPTER)).merge_and_unload()
params = dict(merged.named_parameters())
assert set(params) == set(before), "the merged model's parameter names differ from the starting model's"
changed = {n for n, p in params.items() if fingerprint(p) != before[n]}
CHECKS = {"weights": len(params), "changed": len(changed), "targets": len(TARGETS),
          "changed_outside_targets": sorted(changed - TARGETS), "targets_unchanged": sorted(TARGETS - changed),
          "non_finite": sorted(n for n in changed if not torch.isfinite(params[n]).all())}
print(f"  {len(params):,} weight tensors; changed {len(changed)} (expected {len(TARGETS)}); outside the targets "
      f"{len(CHECKS['changed_outside_targets'])}; targets unchanged {len(CHECKS['targets_unchanged'])}; "
      f"with inf or NaN {len(CHECKS['non_finite'])}")
assert changed == TARGETS and not CHECKS["non_finite"], \
    f"merge check failed, nothing pushed: {({k: v[:5] for k, v in CHECKS.items() if isinstance(v, list)})}"
MERGE_MIN = (time.time() - t0) / 60
print(f"✓ Merged and checked in {MERGE_MIN:.1f} min: exactly the {len(TARGETS)} adapted projections changed")

MERGED_DIR = Path("/tmp/l0s2_merged")          # 16 GB: /kaggle/working keeps only the small outputs
merged.save_pretrained(MERGED_DIR)
AutoProcessor.from_pretrained(START).save_pretrained(MERGED_DIR)   # the stock processor: full size
del base, merged, params
gc.collect()
local_files = {p.relative_to(MERGED_DIR).as_posix(): p.stat().st_size for p in MERGED_DIR.rglob("*") if p.is_file()}

t1 = time.time()
api.create_repo(CFG["merged_repo"], private=not CFG["merged_public"], exist_ok=True)
api.upload_folder(folder_path=str(MERGED_DIR), repo_id=CFG["merged_repo"], delete_patterns="*.safetensors",
                  commit_message=f"L0 step 2: {NAME} merged into our 7B")
remote = {f.path: f.size for f in api.list_repo_tree(CFG["merged_repo"], recursive=True) if hasattr(f, "size")}
mismatched = {k: (v, remote.get(k)) for k, v in local_files.items() if remote.get(k) != v}
assert not mismatched, f"upload check failed (local size, size on the Hub): {mismatched}"
UPLOAD_MIN = (time.time() - t1) / 60
TOTAL_GB = sum(local_files.values()) / 1e9
print(f"✓ Pushed {CFG['merged_repo']} ({len(local_files)} files, {TOTAL_GB:.1f} GB) in {UPLOAD_MIN:.1f} min; "
      "every file on the Hub has its local size")

result = {**CFG, "checkpoint": NAME, "versions": VERSIONS, "trainer_state": {"global_step": STATE["global_step"],
          "max_steps": STATE["max_steps"], "last_losses": LOSSES[-3:]},
          "adapter": {"r": ADAPTER_CFG["r"], "lora_alpha": ADAPTER_CFG["lora_alpha"], "lora_pairs": LORA_PAIRS,
                      "saved_name_patterns": len(ADAPTER_CFG["target_modules"])},
          "checks": CHECKS, "files": len(local_files), "size_gb": round(TOTAL_GB, 2),
          "merge_minutes": round(MERGE_MIN, 1), "upload_minutes": round(UPLOAD_MIN, 1)}
(WORK / "merge_check.json").write_text(json.dumps(result, indent=1), encoding="utf-8")
(WORK / "README.md").write_text("\n".join([
    "# L0 step 2: the merge", "",
    f"`{NAME}` of `{CFG['adapter_repo']}/{CFG['hub_dir']}` (step {STATE['global_step']} of {STATE['max_steps']}, "
    f"LoRA rank {ADAPTER_CFG['r']}, alpha {ADAPTER_CFG['lora_alpha']}) merged into `{CFG['start_model']}` in fp16 "
    f"on the CPU, and pushed as `{CFG['merged_repo']}` ({'public' if CFG['merged_public'] else 'private'}).", "",
    "## Checks", "",
    f"- Weight tensors changed: {len(changed)} of {CHECKS['weights']:,}, exactly the "
    f"{len(TARGETS)} language-model projections the adapter trained (none outside them, none of them unchanged).",
    f"- Changed tensors with inf or NaN: {len(CHECKS['non_finite'])}.",
    f"- Upload: {len(local_files)} files, {TOTAL_GB:.1f} GB; every file on the Hub has its local size.",
    f"- Last three training losses: {LOSSES[-3:]}.", "",
    f"Versions: {VERSIONS}. Merge {MERGE_MIN:.1f} min, upload {UPLOAD_MIN:.1f} min.", "",
    "Next: `l0-step2-full-size-rescore.ipynb` with `MODEL = \"new\"`, in parts A, B and C.",
]), encoding="utf-8")
print(f"✓ Wrote {WORK}: README.md, merge_check.json. Next: l0-step2-full-size-rescore.ipynb with MODEL = \"new\".")

⏳ Merging checkpoint-28 into AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP (fp16, on the CPU, as the first merge was done)...


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

  729 weight tensors; changed 196 (expected 196); outside the targets 0; targets unchanged 0; with inf or NaN 0
✓ Merged and checked in 4.3 min: exactly the 196 adapted projections changed


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Pushed AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP (7 files, 16.6 GB) in 2.4 min; every file on the Hub has its local size
✓ Wrote /kaggle/working/L0S2-merge: README.md, merge_check.json. Next: l0-step2-full-size-rescore.ipynb with MODEL = "new".
